In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_auc_score
import xgboost as xgb
import lightgbm as lgb
from sklearn.cluster import KMeans, DBSCAN
from sklearn.impute import SimpleImputer
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Dropout
import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries installed and imported successfully!")

In [ ]:
# Step 2: Data Acquisition & ETL Operations
print("=== DATA ACQUISITION & ETL OPERATIONS ===")

def load_and_merge_data():
    """Load and merge datasets from specified paths"""
    try:
        # Extract data from CSV files
        train_data = pd.read_csv('customer_churn_dataset-training-master.csv')
        test_data = pd.read_csv('customer_churn_dataset-testing-master.csv')
        
        print(f"✅ Training data loaded: {train_data.shape}")
        print(f"✅ Testing data loaded: {test_data.shape}")
        
        # Merge datasets
        merged_data = pd.concat([train_data, test_data], ignore_index=True)
        print(f"✅ Merged dataset shape: {merged_data.shape}")
        
        return merged_data
        
    except Exception as e:
        print(f"❌ Error loading files: {e}")
        print("📊 Creating sample dataset...")
        return create_sample_data()

def create_sample_data():
    """Create sample data if files not found"""
    np.random.seed(42)
    sample_data = {
        'CustomerID': range(1, 1001),
        'Age': np.random.randint(18, 70, 1000),
        'Gender': np.random.choice(['Male', 'Female'], 1000),
        'Tenure': np.random.randint(1, 60, 1000),
        'Usage Frequency': np.random.randint(1, 30, 1000),
        'Support Calls': np.random.randint(0, 15, 1000),
        'Payment Delay': np.random.randint(0, 35, 1000),
        'Subscription Type': np.random.choice(['Basic', 'Standard', 'Premium'], 1000),
        'Contract Length': np.random.choice(['Monthly', 'Quarterly', 'Annual'], 1000),
        'Total Spend': np.random.uniform(100, 1000, 1000),
        'Last Interaction': np.random.randint(1, 31, 1000),
        'Churn': np.random.choice([0, 1], 1000, p=[0.7, 0.3])
    }
    return pd.DataFrame(sample_data)

# Load data
df = load_and_merge_data()
print("📋 Dataset preview:")
print(df.head())
print(f"\n📊 Dataset info: {df.shape}")

In [ ]:
# Step 3: Data Preprocessing & NaN Handling
print("\n=== DATA PREPROCESSING & NaN HANDLING ===")

# Create a copy for processing
df_processed = df.copy()

# Identify columns with missing values
print("🔍 Checking for missing values:")
print(df_processed.isnull().sum())

# Handle missing values
print("\n🛠️ Handling missing values...")

# Numerical columns - fill with median
numerical_cols = ['Age', 'Tenure', 'Usage Frequency', 'Support Calls', 
                 'Payment Delay', 'Total Spend', 'Last Interaction']

for col in numerical_cols:
    if col in df_processed.columns:
        if df_processed[col].isnull().sum() > 0:
            df_processed[col].fillna(df_processed[col].median(), inplace=True)
            print(f"✅ Filled NaN in {col} with median: {df_processed[col].median():.2f}")

# Categorical columns - fill with mode
categorical_cols = ['Gender', 'Subscription Type', 'Contract Length']
for col in categorical_cols:
    if col in df_processed.columns:
        if df_processed[col].isnull().sum() > 0:
            mode_val = df_processed[col].mode()[0]
            df_processed[col].fillna(mode_val, inplace=True)
            print(f"✅ Filled NaN in {col} with mode: {mode_val}")

# Verify no more missing values
print(f"\n✅ Missing values after processing: {df_processed.isnull().sum().sum()}")

In [ ]:
# Step 4: Feature Engineering & Encoding
print("\n=== FEATURE ENGINEERING & ENCODING ===")

# Feature Engineering
print("🎯 Creating new features...")

# Engagement score
df_processed['Engagement_Score'] = (df_processed['Usage Frequency'] - df_processed['Support Calls']) / df_processed['Tenure'].clip(1)

# Payment behavior
df_processed['Payment_Behavior'] = df_processed['Payment Delay'] / (df_processed['Tenure'] + 1)

# Customer value score
df_processed['Value_Score'] = df_processed['Total Spend'] / df_processed['Tenure'].clip(1)

# Encode categorical variables
print("🔤 Encoding categorical variables...")
label_encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    df_processed[col + '_encoded'] = le.fit_transform(df_processed[col])
    label_encoders[col] = le
    print(f"✅ Encoded {col}")

print("📊 Processed dataset info:")
print(df_processed.info())

In [ ]:
print("\n=== DATA NORMALIZATION & SPLIT ===")

# Drop rows where target 'Churn' is NaN
df_processed = df_processed.dropna(subset=['Churn'])

# Define feature columns
feature_columns = ['Age', 'Tenure', 'Usage Frequency', 'Support Calls', 'Payment Delay', 
                  'Total Spend', 'Last Interaction', 'Gender_encoded', 
                  'Subscription Type_encoded', 'Contract Length_encoded',
                  'Engagement_Score', 'Payment_Behavior', 'Value_Score']

# Remove any columns that might not exist
feature_columns = [col for col in feature_columns if col in df_processed.columns]

X = df_processed[feature_columns]
y = df_processed['Churn']

print(f"📐 Feature matrix shape: {X.shape}")
print(f"🎯 Target variable distribution:\n{y.value_counts()}")

# Normalize numerical data
print("\n📏 Normalizing numerical features...")
scaler = StandardScaler()
X_normalized = scaler.fit_transform(X)

# Split data: 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X_normalized, y, test_size=0.2, random_state=42, stratify=y
)

print(f"✅ Training set: {X_train.shape}")
print(f"✅ Testing set: {X_test.shape}")
print(f"✅ Train Churn distribution: {pd.Series(y_train).value_counts().to_dict()}")
print(f"✅ Test Churn distribution: {pd.Series(y_test).value_counts().to_dict()}")


In [ ]:
# Step 6: Supervised Learning Models with Hyperparameter Tuning
print("\n=== SUPERVISED LEARNING MODELS ===")

# 1. Logistic Regression
print("1. 🔮 LOGISTIC REGRESSION")
lr_params = {'C': [0.1, 1, 10], 'solver': ['liblinear', 'saga']}
lr_grid = GridSearchCV(LogisticRegression(max_iter=1000, random_state=42), 
                       lr_params, cv=5, scoring='accuracy')
lr_grid.fit(X_train, y_train)
best_lr = lr_grid.best_estimator_
lr_score = best_lr.score(X_test, y_test)
print(f"✅ Best params: {lr_grid.best_params_}, Accuracy: {lr_score:.4f}")

# 2. XGBoost
print("\n2. ⚡ XGBOOST")
xgb_params = {'n_estimators': [100, 200], 'learning_rate': [0.05, 0.1]}
xgb_grid = GridSearchCV(xgb.XGBClassifier(random_state=42), 
                        xgb_params, cv=5, scoring='accuracy')
xgb_grid.fit(X_train, y_train)
best_xgb = xgb_grid.best_estimator_
xgb_score = best_xgb.score(X_test, y_test)
print(f"✅ Best params: {xgb_grid.best_params_}, Accuracy: {xgb_score:.4f}")

# 3. LightGBM
print("\n3. 💡 LIGHTGBM")
lgb_params = {'n_estimators': [100, 200], 'learning_rate': [0.05, 0.1]}
lgb_grid = GridSearchCV(lgb.LGBMClassifier(random_state=42), 
                        lgb_params, cv=5, scoring='accuracy')
lgb_grid.fit(X_train, y_train)
best_lgb = lgb_grid.best_estimator_
lgb_score = best_lgb.score(X_test, y_test)
print(f"✅ Best params: {lgb_grid.best_params_}, Accuracy: {lgb_score:.4f}")

# 4. Random Forest
print("\n4. 🌳 RANDOM FOREST")
rf_params = {'n_estimators': [100, 200], 'max_depth': [10, 20]}
rf_grid = GridSearchCV(RandomForestClassifier(random_state=42), 
                       rf_params, cv=5, scoring='accuracy')
rf_grid.fit(X_train, y_train)
best_rf = rf_grid.best_estimator_
rf_score = best_rf.score(X_test, y_test)
print(f"✅ Best params: {rf_grid.best_params_}, Accuracy: {rf_score:.4f}")


In [ ]:
# Step 7: Deep Learning Models (20 Epochs)
print("\n=== DEEP LEARNING MODELS (20 EPOCHS) ===")

# 1. Artificial Neural Network (ANN)
print("1. 🧠 ARTIFICIAL NEURAL NETWORK")
ann_model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dropout(0.3),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid')
])

ann_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Train with 20 epochs
ann_history = ann_model.fit(
    X_train, y_train,
    epochs=20,  # Reduced to 20 epochs
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

ann_pred = (ann_model.predict(X_test) > 0.5).astype("int32")
ann_accuracy = accuracy_score(y_test, ann_pred)
print(f"✅ ANN Accuracy: {ann_accuracy:.4f}")

# 2. RNN for Time-based Trends
print("\n2. 🔄 RECURRENT NEURAL NETWORK")
X_train_rnn = X_train.reshape(X_train.shape[0], 1, X_train.shape[1])
X_test_rnn = X_test.reshape(X_test.shape[0], 1, X_test.shape[1])

rnn_model = Sequential([
    LSTM(50, return_sequences=True, input_shape=(1, X_train.shape[1])),
    Dropout(0.2),
    LSTM(50),
    Dropout(0.2),
    Dense(25, activation='relu'),
    Dense(1, activation='sigmoid')
])

rnn_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Train with 20 epochs
rnn_history = rnn_model.fit(
    X_train_rnn, y_train,
    epochs=20,  # Reduced to 20 epochs
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

rnn_pred = (rnn_model.predict(X_test_rnn) > 0.5).astype("int32")
rnn_accuracy = accuracy_score(y_test, rnn_pred)
print(f"✅ RNN Accuracy: {rnn_accuracy:.4f}")

In [ ]:
# Step 8: Unsupervised Learning & Customer Segmentation
print("\n=== UNSUPERVISED LEARNING ===")

# 1. K-Means Clustering
print("1. 📊 K-MEANS CLUSTERING")
kmeans = KMeans(n_clusters=3, random_state=42)
df_processed['KMeans_Cluster'] = kmeans.fit_predict(X_normalized)
print(f"✅ Customer segments created: {df_processed['KMeans_Cluster'].nunique()}")

# 2. DBSCAN Anomaly Detection
print("\n2. 🚨 DBSCAN ANOMALY DETECTION")
dbscan = DBSCAN(eps=0.5, min_samples=5)
df_processed['DBSCAN_Cluster'] = dbscan.fit_predict(X_normalized)
anomalies = sum(df_processed['DBSCAN_Cluster'] == -1)
print(f"✅ Anomalies detected: {anomalies}")

# Analyze segments
print("\n📈 Segment Analysis:")
print(df_processed.groupby('KMeans_Cluster')['Churn'].mean())

In [ ]:
# Step 9: Model Comparison & Results
print("\n=== MODEL COMPARISON & FINAL RESULTS ===")

# Collect all results
results = {
    'Logistic Regression': lr_score,
    'XGBoost': xgb_score,
    'LightGBM': lgb_score,
    'ANN': ann_accuracy,
    'RNN': rnn_accuracy
}

# Display results
print("🏆 MODEL PERFORMANCE SUMMARY:")
print("=" * 50)
for model, accuracy in results.items():
    print(f"{model:<20} | Accuracy: {accuracy:.4f}")

# Best model
best_model_name = max(results, key=results.get)
best_accuracy = results[best_model_name]
print("=" * 50)
print(f"🎯 BEST MODEL: {best_model_name}")
print(f"🏅 BEST ACCURACY: {best_accuracy:.4f}")

# Feature Importance
print("\n🔍 TOP 10 FEATURE IMPORTANCE:")
feature_importance = pd.DataFrame({
    'feature': feature_columns,
    'importance': best_xgb.feature_importances_
}).sort_values('importance', ascending=False)

print(feature_importance.head(10))

print("\n✅ PROJECT COMPLETED SUCCESSFULLY!")
print("📊 All models trained with 20 epochs and NaN values handled!")